# 1. Purpose and scope

This notebook generates standardized Phase 2 candidate signals from clean OHLCV panels stored in SQLite. It creates candidates, structural quality gates, metadata, and family summaries only; it does not select final signals or run walk-forward validation.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

cwd = Path.cwd().resolve()
project_root = next((path for path in [cwd, *cwd.parents] if (path / 'src').exists()), None)
if project_root is None:
    raise RuntimeError('Could not locate project root from current working directory.')

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


# 2. Imports and config

In [2]:
from src.db import get_db_path, load_ohlcv_panels, load_table, table_exists
from src.run_config import make_run_id, make_run_timestamp
from src.signal_quality import (
    APPROVED_FOR_SCORING,
    REJECTED_DATA_QUALITY,
    WATCHLIST,
    build_signal_family_summary,
    build_signal_quality_summary,
    filter_signal_quality,
)
from src.signal_storage import SIGNAL_TABLES, save_signal_factory_outputs
from src.signals import generate_signal_library

SIGNAL_VERSION = 'phase2_candidate_v1'
DISCOVERY_VERSION = 'phase2_discovery_generated_v1'
MAX_DISCOVERY_SIGNALS = 30
DISCOVERY_SEARCH_SPACE_TABLE = 'signal_discovery_search_space_current'
sqlite_db_path = get_db_path()

print(f'SQLite database: {sqlite_db_path}')
print(f'Signal version: {SIGNAL_VERSION}')
print(f'Discovery version: {DISCOVERY_VERSION}')
print(f'Max discovery signals: {MAX_DISCOVERY_SIGNALS}')


SQLite database: /Users/AnyiXu_1/Desktop/multi-factor-equity-alpha-model/sql/project_underdog.db
Signal version: phase2_candidate_v1
Discovery version: phase2_discovery_generated_v1
Max discovery signals: 30


# 3. Create run_id / timestamp

In [3]:
run_id = make_run_id(prefix='phase2_nb02')
run_timestamp = make_run_timestamp()

print(f'run_id: {run_id}')
print(f'run_timestamp: {run_timestamp}')


run_id: phase2_nb02_20260507_224108
run_timestamp: 2026-05-07 22:41:08


# 4. Load OHLCV panels from SQLite

In [4]:
panels = load_ohlcv_panels(current=True, db_path=sqlite_db_path)

panel_shapes = pd.Series({name: panel.shape for name, panel in panels.items()}, name='shape')
display(panel_shapes)
display(panels['close'].head())


open      (2098, 478)
high      (2098, 478)
low       (2098, 478)
close     (2098, 478)
volume    (2098, 478)
Name: shape, dtype: object

,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,WTW,WY,WYNN,XEL,XOM,XYL,YUM,ZBH,ZBRA,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 5. Validate panel shapes

In [5]:
shape_set = {panel.shape for panel in panels.values()}
if len(shape_set) != 1:
    raise ValueError(f'OHLCV panels do not share one shape: {panel_shapes.to_dict()}')

index_set = {tuple(panel.index) for panel in panels.values()}
column_set = {tuple(panel.columns) for panel in panels.values()}
if len(index_set) != 1 or len(column_set) != 1:
    raise ValueError('OHLCV panels do not share identical Date indexes and ticker columns.')

metadata_columns_present = {
    name: sorted(set(panel.columns).intersection({'run_id', 'run_timestamp', 'timestamp'}))
    for name, panel in panels.items()
}
if any(metadata_columns_present.values()):
    raise ValueError(f'Metadata columns found in price panels: {metadata_columns_present}')

print(f'Validated {len(panels)} panels with shape {next(iter(shape_set))}.')


Validated 5 panels with shape (2098, 478).


# 6. Generate candidate signals

In [6]:
if not table_exists(DISCOVERY_SEARCH_SPACE_TABLE, db_path=sqlite_db_path):
    raise ValueError(f'Missing required discovery search-space table: {DISCOVERY_SEARCH_SPACE_TABLE}')

discovery_search_space = load_table(DISCOVERY_SEARCH_SPACE_TABLE, db_path=sqlite_db_path)
display(discovery_search_space)

signals, metadata = generate_signal_library(
    panels=panels,
    signal_version=SIGNAL_VERSION,
    run_id=run_id,
    timestamp=run_timestamp,
    discovery_search_space=discovery_search_space,
    discovery_version=DISCOVERY_VERSION,
    max_discovery_signals=MAX_DISCOVERY_SIGNALS,
)

signal_shapes = pd.Series({name: signal.shape for name, signal in signals.items()}, name='shape')
display(signal_shapes)
display(next(iter(signals.values())).head())


,discovery_family,base_formula,signal_template_name,parameter_grid,required_inputs,transform_options,direction_hypotheses,expected_horizons,expected_diversification_role,priority,notes,run_id,search_space_version,timestamp
0,cross_sectional_relative_return,close.pct_change(window) - cross_sectional_mea...,relative_return_{window},"{""directions"":[""positive_edge"",""negative_edge_...","[""close""]","[""raw"",""rank"",""zscore"",""winsorized_zscore""]","[""positive_edge"",""negative_edge_reverse""]","[""5d"",""10d"",""20d""]",Adds same-date universe-relative price informa...,HIGH,Use trailing returns only; center by same-date...,phase2_signal_discovery_space_20260503_204034,phase2_signal_discovery_space_v1,2026-05-03 20:40:34
1,beta_neutral_return,close.pct_change(window) - rolling_beta(beta_w...,beta_neutral_return_{window}_{beta_window},"{""beta_windows"":[60,120],""directions"":[""positi...","[""close"",""benchmark_close""]","[""raw"",""rank"",""zscore"",""winsorized_zscore""]","[""positive_edge"",""negative_edge_reverse""]","[""5d"",""10d"",""20d""]",Separates stock-specific return from broad sys...,HIGH,Use SPY when present; otherwise equal-weight u...,phase2_signal_discovery_space_20260503_204034,phase2_signal_discovery_space_v1,2026-05-03 20:40:34
2,volatility_adjusted_momentum,close.pct_change(window) / rolling_std(daily_r...,vol_adj_momentum_{window}_{vol_window},"{""directions"":[""positive_edge"",""negative_edge_...","[""close""]","[""raw"",""rank"",""zscore"",""winsorized_zscore""]","[""positive_edge"",""negative_edge_reverse""]","[""10d"",""20d""]",Tests smoother momentum variants without relyi...,MEDIUM,Handle zero realized volatility as missing bef...,phase2_signal_discovery_space_20260503_204034,phase2_signal_discovery_space_v1,2026-05-03 20:40:34
3,volatility_surprise,"rolling_std(daily_return, short_window) / roll...",vol_surprise_{short_window}_{long_window},"{""directions"":[""positive_edge"",""negative_edge_...","[""close""]","[""raw"",""rank"",""zscore"",""winsorized_zscore""]","[""positive_edge"",""negative_edge_reverse""]","[""5d"",""10d"",""20d""]",Captures volatility regime transitions rather ...,MEDIUM,Treat as volatility-adjacent and require diver...,phase2_signal_discovery_space_20260503_204034,phase2_signal_discovery_space_v1,2026-05-03 20:40:34
4,volume_return_interaction,"return(window) combined with volume trend, acc...",volume_return_interaction_{window}_{interaction},"{""directions"":[""positive_edge"",""negative_edge_...","[""close"",""volume""]","[""raw"",""rank"",""zscore"",""winsorized_zscore""]","[""positive_edge"",""negative_edge_reverse""]","[""5d"",""10d"",""20d""]",Adds participation context to price moves and ...,HIGH,Use trailing volume windows only; replace zero...,phase2_signal_discovery_space_20260503_204034,phase2_signal_discovery_space_v1,2026-05-03 20:40:34
5,reversal_overextension,"negative return(window), distance from moving ...",reversal_overextension_{window}_{measure},"{""directions"":[""positive_edge"",""negative_edge_...","[""close""]","[""raw"",""rank"",""zscore"",""winsorized_zscore""]","[""positive_edge"",""negative_edge_reverse""]","[""1d"",""5d"",""10d""]",Expands short-horizon pullback behavior beyond...,MEDIUM,Keep formulas trailing-only; execution lag rem...,phase2_signal_discovery_space_20260503_204034,phase2_signal_discovery_space_v1,2026-05-03 20:40:34
6,correlation_change,"rolling_corr(stock_return, benchmark_return, s...",corr_change_{short_window}_{long_window},"{""directions"":[""positive_edge"",""negative_edge_...","[""close"",""benchmark_close""]","[""raw"",""rank"",""zscore"",""winsorized_zscore""]","[""positive_edge"",""negative_edge_reverse""]","[""10d"",""20d""]",Targets changing market co-movement and crowdi...,HIGH,"Use SPY if available, otherwise equal-weight m...",phase2_signal_discovery_space_20260503_204034,phase2_signal_discovery_space_v1,2026-05-03 20:40:34
7,liquidity_adjusted_return,return(window) scaled or conditioned by dollar...,liqui

/Users/AnyiXu_1/Desktop/multi-factor-equity-alpha-model/src/signals.py:677: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  return close.pct_change(lookback)
/Users/AnyiXu_1/Desktop/multi-factor-equity-alpha-model/src/signals.py:677: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  return close.pct_change(lookback)
/Users/AnyiXu_1/Desktop/multi-factor-equity-alpha-model/src/signals.py:695: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fil

momentum_20                                           (2098, 478)
momentum_60                                           (2098, 478)
mean_reversion_5                                      (2098, 478)
mean_reversion_20                                     (2098, 478)
volatility_20                                         (2098, 478)
                                                         ...     
disc_beta_neutral_w5_b60_raw                          (2098, 478)
disc_corr_change_s5_l60_raw                           (2098, 478)
disc_volret_interact_w5_returnminusvolumetrend_raw    (2098, 478)
disc_voladj_mom_w5_v20_raw                            (2098, 478)
disc_vol_surprise_s5_l60_raw                          (2098, 478)
Name: shape, Length: 85, dtype: object

,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,WTW,WY,WYNN,XEL,XOM,XYL,YUM,ZBH,ZBRA,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 7. Build metadata

In [7]:
display(metadata)
display(metadata['signal_family'].value_counts().rename('signal_count'))


,signal_name,signal_family,formula_type,parameters,data_dependencies,lookback,direction_convention,input_fields,normalization_notes,normalization,signal_source,discovery_family,discovery_version,signal_template_name,parameter_config_json,signal_version,run_id,timestamp,created_timestamp,notes
0,momentum_20,momentum,close_pct_change,lookback=20,close,20,higher_is_more_bullish,close,Raw trailing signal is cross-sectionally z-sco...,cross_sectional_zscore_by_date,manual_core,,,,,phase2_candidate_v1,phase2_nb02_20260507_224108,2026-05-07 22:41:08,2026-05-07 22:41:08,"20-day close-to-close return, cross-sectionall..."
1,momentum_60,momentum,close_pct_change,lookback=60,close,60,higher_is_more_bullish,close,Raw trailing signal is cross-sectionally z-sco...,cross_sectional_zscore_by_date,manual_core,,,,,phase2_candidate_v1,phase2_nb02_20260507_224108,2026-05-07 22:41:08,2026-05-07 22:41:08,"60-day close-to-close return, cross-sectionall..."
2,mean_reversion_5,mean_reversion,negative_distance_to_moving_average,lookback=5,close,5,higher_is_more_oversold,close,Raw trailing signal is cross-sectionally z-sco...,cross_sectional_zscore_by_date,manual_core,,,,,phase2_candidate_v1,phase2_nb02_20260507_224108,2026-05-07 22:41:08,2026-05-07 22:41:08,Negative price distance from 5-day moving aver...
3,mean_reversion_20,mean_reversion,negative_distance_to_moving_average,lookback=20,close,20,higher_is_more_oversold,close,Raw trailing signal is cross-sectionally z-sco...,cross_sectional_zscore_by_date,manual_core,,,,,phase2_candidate_v1,phase2_nb02_20260507_224108,2026-05-07 22:41:08,2026-05-07 22:41:08,Negative price distance from 20-day moving ave...
4,volatility_20,volatility,realized_return_volatility,lookback=20,close,20,higher_is_more_volatile,close,Raw trailing signal is cross-sectionally z-sco...,cross_sectional_zscore_by_date,manual_core,,,,,phase2_candidate_v1,phase2_nb02_20260507_224108,2026-05-07 22:41:08,2026-05-07 22:41:08,20-day realized close-to-close return volatility.
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80,disc_beta_neutral_w5_b60_raw,beta_neutral_return,discovery_beta_neutral_return,"beta_window=60,transform=raw,window=5",close,60,higher_follows_discovery_hypothesis,close,Discovery raw signal uses transform=raw; final...,cross_sectional_zscore_by_date,discovery_generated,beta_neutral_return,phase2_discovery_generated_v1,beta_neutral_return_{window}_{beta_window},"{""beta_window"":60,""transform"":""raw"",""window"":5}",phase2_candidate_v1,phase2_nb02_20260507_224108,2026-05-07 22:41:08,2026-05-07 22:41:08,Generated from 01C signal discovery search space.
81,disc_corr_change_s5_l60_raw,correlation_change,discovery_correlation_change,"long_window=60,short_window=5,transform=raw",close,60,higher_follows_discovery_hypothesis,close,Discovery raw signal uses transform=raw; final...,cross_sectional_zscore_by_date,discovery_generated,correlation_change,phase2_discovery_generated_v1,corr_change_{short_window}_{long_window},"{""long_window"":60,""short_window"":5,""transform""...",phase2_candidate_v1,phase2_nb02_20260507_224108,2026-05-07 22:41:08,2026-05-07 22:41:08,Generated from 01C signal discovery search space.
82,disc_volret_interact_w5_returnminusvolumetrend...,volume_return_interaction,discovery_volume_return_interaction,"interaction=return_minus_volume_trend,transfor...","close,volume",5,higher_follows_discovery_hypothesis,"close,volume",Discovery raw signal uses transform=raw; final...,cross_sectional_zscore_by_date,discovery_generated,volume_return_interaction,phase2_discovery_generated_v1,volume_return_interaction_{window}_{interaction},"{""interaction"":""return_minus_volume_trend"",""tr...",phase2_candidate_v1,phase2_nb02_20260507_224108,2026-05-07 22:41:08,2026-05-07 22:41:08,Generated from 01C signal discovery search space.
83,disc_voladj_mom_w5_v20_raw,volatility_adjusted_momentum,discovery_volatility_adjusted_momentum,"transform=raw,vol_window=20,window=5",close,20,higher_fo

signal_family
trend_quality                        7
breakout                             6
short_term_reversal                  6
volume_liquidity                     4
volatility_surprise                  4
volatility_adjusted_momentum         4
volume_return_interaction            4
correlation_change                   4
beta_neutral_return                  4
cross_sectional_relative_return      4
defensive_stability                  4
cross_sectional_relative_strength    3
reversal_overextension               3
liquidity_adjusted_return            3
residual_momentum                    3
mean_reversion                       3
correlation_dispersion               3
volume_flow                          3
defensive_quality                    3
volatility_normalized_momentum       2
volume                               2
volatility                           2
momentum                             2
cross_sectional_relative_value       1
liquidity                            1
Name: signa

# 8. Run signal quality checks

In [8]:
quality = build_signal_quality_summary(
    signals=signals,
    metadata=metadata,
    run_id=run_id,
    signal_version=SIGNAL_VERSION,
)

approved_quality, rejected_quality = filter_signal_quality(quality)
quality_gate = pd.concat([approved_quality, rejected_quality], ignore_index=True)
family_summary = build_signal_family_summary(quality)
quality_sorted_by_finite_pct = quality.sort_values(
    ["finite_pct", "missing_pct", "signal_name"],
    ascending=[True, False, True],
).reset_index(drop=True)
failing_quality_checks = quality_gate.loc[
    quality_gate["status"].ne(APPROVED_FOR_SCORING)
].sort_values(["status", "finite_pct", "signal_name"]).reset_index(drop=True)
manual_metadata = metadata.loc[metadata["signal_source"].eq("manual_core")].copy()
generated_metadata = metadata.loc[metadata["signal_source"].eq("discovery_generated")].copy()
generated_signal_names = generated_metadata["signal_name"].sort_values().tolist()
generated_signal_quality = quality.loc[quality["signal_name"].isin(generated_signal_names)].sort_values("signal_name")
generated_all_nan_signals = [
    signal_name
    for signal_name in generated_signal_names
    if signal_name in signals and signals[signal_name].notna().sum().sum() == 0
]
manual_core_signals_present = sorted(set(signals).intersection(manual_metadata["signal_name"]))
source_counts = metadata["signal_source"].value_counts(dropna=False).rename("signal_count")

status_counts = (
    quality_gate['status']
    .value_counts()
    .reindex([APPROVED_FOR_SCORING, WATCHLIST, REJECTED_DATA_QUALITY], fill_value=0)
)

print(f"Total candidate signals: {len(signals)}")
print(f"Manual/core signal count: {len(manual_core_signals_present)}")
print(f"Generated discovery signal count: {len(generated_signal_names)}")
print(f"Generated all-NaN signals: {generated_all_nan_signals}")
print("Count by signal_source")
display(source_counts)
print("Count by signal_family")
display(metadata['signal_family'].value_counts().sort_index().rename('signal_count'))

print("Quality status counts")
display(status_counts.rename('signal_count'))

print("Generated signal metadata rows")
display(generated_metadata)

print("Generated signal quality rows")
display(generated_signal_quality)

print("Quality table sorted by finite_pct")
display(quality_sorted_by_finite_pct)

print("Signals failing quality checks")
display(failing_quality_checks[["signal_name", "signal_family", "status", "finite_pct", "missing_pct", "quality_gate_notes"]])

Total candidate signals: 85
Manual/core signal count: 55
Generated discovery signal count: 30
Generated all-NaN signals: []
Count by signal_source


signal_source
manual_core            55
discovery_generated    30
Name: signal_count, dtype: int64

Count by signal_family


signal_family
beta_neutral_return                  4
breakout                             6
correlation_change                   4
correlation_dispersion               3
cross_sectional_relative_return      4
cross_sectional_relative_strength    3
cross_sectional_relative_value       1
defensive_quality                    3
defensive_stability                  4
liquidity                            1
liquidity_adjusted_return            3
mean_reversion                       3
momentum                             2
residual_momentum                    3
reversal_overextension               3
short_term_reversal                  6
trend_quality                        7
volatility                           2
volatility_adjusted_momentum         4
volatility_normalized_momentum       2
volatility_surprise                  4
volume                               2
volume_flow                          3
volume_liquidity                     4
volume_return_interaction            4
Name: signa

Quality status counts


...

,signal_name,signal_family,n_dates,n_tickers,missing_pct,finite_pct,first_valid_date,last_valid_date,run_id,signal_version
56,disc_beta_neutral_w5_b60_rank,beta_neutral_return,2098,478,0.124010,0.875990,2018-03-29,2026-05-07,phase2_nb02_20260507_224108,phase2_candidate_v1
80,disc_beta_neutral_w5_b60_raw,beta_neutral_return,2098,478,0.124025,0.875975,2018-04-20,2026-05-07,phase2_nb02_20260507_224108,phase2_candidate_v1
72,disc_beta_neutral_w5_b60_wz,beta_neutral_return,2098,478,0.124025,0.875975,2018-04-20,2026-05-07,phase2_nb02_20260507_224108,phase2_candidate_v1
64,disc_beta_neutral_w5_b60_z,beta_neutral_return,2098,478,0.124025,0.875975,2018-04-20,2026-05-07,phase2_nb02_20260507_224108,phase2_candidate_v1
57,disc_corr_change_s5_l60_rank,correlation_change,2098,478,0.381009,0.618991,2018-03-29,2026-05-07,phase2_nb02_20260507_224108,phase2_candidate_v1
81,disc_corr_change_s5_l60_raw,correlation_change,2098,478,0.381024,0.618976,2018-04-20,2026-05-07,phase2_nb02_20260507_224108,phase2_candidate_v1
73,disc_corr_change_s5_l60_wz,correlation_change,2098,478,0.381024,0.618976,2018-04-20,2026-05-07,phase2_nb02_20260507_224108,phase2_candidate_v1
65,disc_corr_change_s5_l60_z,correlation_change,2098,478,0.381024,0.618976,2018-04-20,2026-05-07,phase2_nb02_20260507_224108,phase2_candidate_v1
62,disc_liq_adj_ret_w5_dollarvolumerank_rank,liquidity_adjusted_return,2098,478,0.393018,0.606982,2018-01-09,2026-05-07,phase2_nb02_20260507_224108,phase2_candidate_v1
78,disc_liq_adj_ret_w5_dollarvolumerank_wz,liquidity_adjusted_return,2098,478,0.393033,0.606967,2018-01-31,2026-05-07,phase2_nb02_20260507_224108,phase2_candidate_v1


Quality table sorted by finite_pct


,signal_name,signal_family,n_dates,n_tickers,missing_pct,finite_pct,first_valid_date,last_valid_date,run_id,signal_version
0,price_above_ma_200,trend_quality,2098,478,0.580161,0.419839,2018-11-06,2026-05-07,phase2_nb02_20260507_224108,phase2_candidate_v1
1,trend_strength_50_200,trend_quality,2098,478,0.580161,0.419839,2018-11-06,2026-05-07,phase2_nb02_20260507_224108,phase2_candidate_v1
2,breakout_up_60,breakout,2098,478,0.543075,0.456925,2018-04-20,2026-05-07,phase2_nb02_20260507_224108,phase2_candidate_v1
3,price_above_ma_100,trend_quality,2098,478,0.526910,0.473090,2018-06-15,2026-05-07,phase2_nb02_20260507_224108,phase2_candidate_v1
4,breakout_60,breakout,2098,478,0.494885,0.505115,2018-04-19,2026-05-07,phase2_nb02_20260507_224108,phase2_candidate_v1
...,...,...,...,...,...,...,...,...,...,...
80,disc_reversal_overext_w5_return_z,reversal_overextension,2098,478,0.098687,0.901313,2018-01-31,2026-05-07,phase2_nb02_20260507_224108,phase2_candidate_v1
81,reversal_5d,short_term_reversal,2098,478,0.098687,0.901313,2018-01-31,2026-05-07,phase2_nb02_20260507_224108,phase2_candidate_v1
82,disc_relret_w5_rank,cross_sectional_relative_return,2098,478,0.098672,0.901328,2018-01-09,2026-05-07,phase2_nb02_20260507_224108,phase2_candidate_v1
83,disc_reversal_overext_w5_return_rank,reversal_overextension,2098,478,0.098672,0.901328,2018-01-09,2026-05-07,phase2_nb02_20260507_224108,phase2_candidate_v1


Signals failing quality checks


,signal_name,signal_family,status,finite_pct,missing_pct,quality_gate_notes
0,price_above_ma_200,trend_quality,REJECTED_DATA_QUALITY,0.419839,0.580161,Fails structural data quality thresholds.
1,trend_strength_50_200,trend_quality,REJECTED_DATA_QUALITY,0.419839,0.580161,Fails structural data quality thresholds.
2,breakout_up_60,breakout,REJECTED_DATA_QUALITY,0.456925,0.543075,Fails structural data quality thresholds.
3,price_above_ma_100,trend_quality,REJECTED_DATA_QUALITY,0.473090,0.526910,Fails structural data quality thresholds.
4,breakout_60,breakout,REJECTED_DATA_QUALITY,0.505115,0.494885,Fails structural data quality thresholds.
...,...,...,...,...,...,...
80,disc_reversal_overext_w5_return_z,reversal_overextension,WATCHLIST,0.901313,0.098687,Near threshold; keep visible but exclude from ...
81,reversal_5d,short_term_reversal,WATCHLIST,0.901313,0.098687,Near threshold; keep visible but exclude from ...
82,disc_relret_w5_rank,cross_sectional_relative_return,WATCHLIST,0.901328,0.098672,Near threshold; keep visible but exclude from ...
83,disc_reversal_overext_w5_return_rank,reversal_overextension,WATCHLIST,0.901328,0.098672,Near threshold; keep visible but exclude from ...


# 9. Save signals, metadata, and quality summary to SQLite

In [9]:
saved_paths = save_signal_factory_outputs(
    signals=signals,
    metadata=metadata,
    quality=quality,
    quality_gate=quality_gate,
    family_summary=family_summary,
    db_path=sqlite_db_path,
    run_id=run_id,
    signal_version=SIGNAL_VERSION,
)

sqlite_tables_written = pd.DataFrame(
    [
        {
            'artifact': artifact,
            'current_table': tables[0],
            'history_table': tables[1],
            'sqlite_path': str(saved_paths[artifact]),
        }
        for artifact, tables in SIGNAL_TABLES.items()
    ]
)

display(sqlite_tables_written)

,artifact,current_table,history_table,sqlite_path
0,signals,candidate_signals_current,candidate_signals_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
1,metadata,candidate_signal_metadata_current,candidate_signal_metadata_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
2,quality,candidate_signal_quality_current,candidate_signal_quality_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
3,quality_gate,candidate_signal_quality_gate_current,candidate_signal_quality_gate_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
4,family_summary,candidate_signal_family_summary_current,candidate_signal_family_summary_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...


# 10. Final summary

In [10]:
summary = pd.DataFrame([
    {
        'run_id': run_id,
        'signal_version': SIGNAL_VERSION,
        'total_signals_generated': len(signals),
        'manual_core_signal_count': len(manual_core_signals_present),
        'generated_discovery_signal_count': len(generated_signal_names),
        'generated_all_nan_signals_count': len(generated_all_nan_signals),
        'approved_for_scoring_count': int(status_counts.loc[APPROVED_FOR_SCORING]),
        'watchlist_count': int(status_counts.loc[WATCHLIST]),
        'rejected_count': int(status_counts.loc[REJECTED_DATA_QUALITY]),
        'n_dates': next(iter(signals.values())).shape[0],
        'n_tickers': next(iter(signals.values())).shape[1],
        'sqlite_db_path': str(sqlite_db_path),
    }
])

print("Final summary")
display(summary)

print("Total signal count")
print(len(signals))

print("Manual/core signal count")
print(len(manual_core_signals_present))

print("Generated discovery signal count")
print(len(generated_signal_names))

print("Generated discovery signals")
display(pd.DataFrame({'signal_name': generated_signal_names}))

print("Generated signals rejected as all-NaN")
display(pd.DataFrame({'signal_name': generated_all_nan_signals}))

print("Quality status counts")
display(status_counts.rename('signal_count'))

print("Generated signal metadata rows")
display(generated_metadata)

print("Generated signal quality rows")
display(generated_signal_quality)

print("Source counts")
display(source_counts)

print("Family counts")
display(metadata['signal_family'].value_counts().sort_index().rename('signal_count'))

print("Family summary")
display(family_summary)

print("SQLite tables written")
display(sqlite_tables_written)

Final summary


,run_id,signal_version,total_signals_generated,manual_core_signal_count,generated_discovery_signal_count,generated_all_nan_signals_count,approved_for_scoring_count,watchlist_count,rejected_count,n_dates,n_tickers,sqlite_db_path
0,phase2_nb02_20260507_224108,phase2_candidate_v1,85,55,30,0,0,29,56,2098,478,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...


Total signal count
85
Manual/core signal count
55
Generated discovery signal count
30
Generated discovery signals


,signal_name
0,disc_beta_neutral_w5_b60_rank
1,disc_beta_neutral_w5_b60_raw
2,disc_beta_neutral_w5_b60_wz
3,disc_beta_neutral_w5_b60_z
4,disc_corr_change_s5_l60_rank
5,disc_corr_change_s5_l60_raw
6,disc_corr_change_s5_l60_wz
7,disc_corr_change_s5_l60_z
8,disc_liq_adj_ret_w5_dollarvolumerank_rank
9,disc_liq_adj_ret_w5_dollarvolumerank_wz


Generated signals rejected as all-NaN


...

signal_family
beta_neutral_return                  4
breakout                             6
correlation_change                   4
correlation_dispersion               3
cross_sectional_relative_return      4
cross_sectional_relative_strength    3
cross_sectional_relative_value       1
defensive_quality                    3
defensive_stability                  4
liquidity                            1
liquidity_adjusted_return            3
mean_reversion                       3
momentum                             2
residual_momentum                    3
reversal_overextension               3
short_term_reversal                  6
trend_quality                        7
volatility                           2
volatility_adjusted_momentum         4
volatility_normalized_momentum       2
volatility_surprise                  4
volume                               2
volume_flow                          3
volume_liquidity                     4
volume_return_interaction            4
Name: signa

Family summary


,signal_family,n_signals,avg_missing_pct,avg_finite_pct,min_first_valid_date,max_first_valid_date
0,beta_neutral_return,4,0.124022,0.875978,2018-03-29,2018-04-20
1,breakout,6,0.479528,0.520472,2018-02-21,2018-04-20
2,correlation_change,4,0.381021,0.618979,2018-03-29,2018-04-20
3,correlation_dispersion,3,0.184538,0.815462,2018-04-20,2018-05-18
4,cross_sectional_relative_return,4,0.098684,0.901316,2018-01-09,2018-01-31
5,cross_sectional_relative_strength,3,0.127082,0.872918,2018-01-31,2018-06-25
6,cross_sectional_relative_value,1,0.105598,0.894402,2018-02-22,2018-02-22
7,defensive_quality,3,0.243949,0.756051,2018-02-22,2018-04-20
8,defensive_stability,4,0.213968,0.786032,2018-02-22,2018-04-20
9,liquidity,1,0.438266,0.561734,2018-02-22,2018-02-22


SQLite tables written


,artifact,current_table,history_table,sqlite_path
0,signals,candidate_signals_current,candidate_signals_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
1,metadata,candidate_signal_metadata_current,candidate_signal_metadata_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
2,quality,candidate_signal_quality_current,candidate_signal_quality_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
3,quality_gate,candidate_signal_quality_gate_current,candidate_signal_quality_gate_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
4,family_summary,candidate_signal_family_summary_current,candidate_signal_family_summary_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
